# calo-ddpm inpainting — step-by-step experimentation

Interactive walkthrough of the full pipeline: model loading → schedule →
event generation → paper observables → noise-free inpainting with all five
algorithms → posterior visualization → mini statistical analysis (SBC /
coverage / pulls / TARP) → DDIM η sweep.

**Prerequisites:** repo installed env (`conda activate improved-diffusion`),
pre-trained weights unpacked (see `scripts/download_weights.sh`).

**Speed knobs** (cell below): everything here uses `S_FAST` sampling steps for
quick turnaround. For paper-quality numbers use `S_FAST = 8000` (generation)
/ `1000` (inpainting) — or better, the batch scripts in `scripts/`.
All knobs can also be overridden via environment variables.

In [ ]:
import os, sys, time, json
sys.path.insert(0, '..')                       # repo root

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm as MplLogNorm

# ---- knobs (env-overridable) ------------------------------------------------
ROOT      = os.environ.get('ROOT', '/home/roli/DiffusionProject')
CENT      = os.environ.get('CENT', 'cent0')
MODEL_DIR = os.path.join(ROOT, 'pre-trained-model-weights', f'{CENT}_ddpm_seed0')
EVENTS    = os.path.join(ROOT, 'generated_events', f'events_{CENT}_seed0.npy')

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
S_FAST        = int(os.environ.get('S_FAST', 250))    # sampling steps here
N_GEN         = int(os.environ.get('N_GEN', 16))      # events to generate
N_POST        = int(os.environ.get('N_POST', 20))     # posterior samples/image
N_IMAGES_MINI = int(os.environ.get('N_IMAGES_MINI', 16))  # mini SBC images
BOX, ETA0, PHI0 = 4, 8, 28                             # dead region
USE_BF16      = DEVICE == 'cuda'

print(f'device={DEVICE}  model={MODEL_DIR}  S_FAST={S_FAST}')

## 1. Load the pre-trained model and build the schedule

`load_model` parses the jetgen `config.json` (so β-schedule and architecture
are always the right ones for the centrality) and strips the `_net.` prefix
from the EMA checkpoint `net_avg_gen.pth`.

In [ ]:
from calo_inpaint.ddpm_sampler import (
    load_model, schedule_from_config, lognorm_from_config, DDPMSampler)
from calo_inpaint.ddim_sampler import DDIMSampler

net, config = load_model(MODEL_DIR, DEVICE)
sched   = schedule_from_config(config, S=S_FAST, device=DEVICE)
lognorm = lognorm_from_config(config)

vs = config['model_args']['vsched']
print(f"T={vs['T']}  beta1={vs['beta1']:.3g}  betaT={vs['betaT']:.3g}")
print(f"UNet params: {sum(p.numel() for p in net.parameters()):,}")
print(f"scale_shift_norm: {config['generator']['model_args']['use_scale_shift_norm']}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(sched.t_map.cpu()[1:], sched.step_var.cpu()[1:]); ax[0].set_yscale('log')
ax[0].set_xlabel('t'); ax[0].set_title('subsampled step variance')
ax[1].plot(sched.t_map.cpu(), sched.sbar.cpu(), label=r'$\bar s_t$')
ax[1].plot(sched.t_map.cpu(), sched.vbar.cpu(), label=r'$\bar v_t$')
ax[1].legend(); ax[1].set_xlabel('t'); ax[1].set_title('cumulative schedule')
plt.tight_layout()

## 2. Generate a few events

Ancestral DDPM in log space, then `exp` back to GeV. (Paper setup:
`S = T = 8000`; small `S_FAST` gives visually fine but slightly off-spectrum
events — good enough for pipeline experimentation.)

In [ ]:
sampler = DDPMSampler(net, sched, DEVICE, seed=0, use_bf16=USE_BF16)

t0 = time.time()
with torch.no_grad():
    x_log = sampler.sample(N_GEN)               # (N, 1, 24, 64) log space
gen_gev = lognorm.denormalize(x_log).squeeze(1).cpu().numpy()
print(f'{N_GEN} events in {time.time()-t0:.1f}s   '
      f'<SumET> = {gen_gev.sum(axis=(1,2)).mean():.1f} GeV')

fig, axes = plt.subplots(2, 2, figsize=(11, 4))
for a, ev in zip(axes.ravel(), gen_gev):
    im = a.imshow(np.clip(ev, 1e-3, None), norm=MplLogNorm(1e-3, 10),
                  aspect='auto', cmap='inferno')
    a.set_xlabel(r'$\phi$'); a.set_ylabel(r'$\eta$')
fig.colorbar(im, ax=axes, label=r'$E_T$ [GeV]', shrink=0.8);

## 3. Verification observables (paper Fig. 4)

Tower-area spectra and ΣE_T. Uses the pre-generated event file if present
(recommended: run the `generate` stage first), else the events from above.

In [ ]:
from calo_inpaint.metrics import window_sums, sum_et, TOWER_AREAS

if os.path.exists(EVENTS):
    events = np.asarray(np.load(EVENTS, mmap_mode='r')[:20000], dtype=np.float32)
    print(f'using {EVENTS}: {events.shape}')
else:
    events = gen_gev
    print('event file not found — using the freshly generated batch')

fig, axes = plt.subplots(1, len(TOWER_AREAS)+1, figsize=(16, 3))
for ax, k in zip(axes, TOWER_AREAS):
    v = window_sums(events, k).ravel()
    ax.hist(v, bins=60, histtype='step', density=True)
    ax.set_yscale('log'); ax.set_xlabel(f'{k}x{k} $E_T$ [GeV]')
tot = sum_et(events)
axes[-1].hist(tot, bins=40, histtype='step', density=True)
axes[-1].set_xlabel(r'$\Sigma E_T$ [GeV]')
axes[0].set_ylabel('(1/N) dN/dE')
print(f'<SumET> = {tot.mean():.1f} ± {tot.std():.1f} GeV')
plt.tight_layout()

## 4. Dead region: truth, mask, measurement

Noise-free setting: `y = mask * x0_true` in log space; mask=1 on live pixels.

In [ ]:
from calo_inpaint.masks import square_mask

truth_gev = torch.from_numpy(events[0:1].copy()).to(DEVICE)   # (1, 24, 64)
y_log     = lognorm.normalize(truth_gev)
mask      = square_mask(BOX, ETA0, PHI0, device=DEVICE)

fig, ax = plt.subplots(1, 2, figsize=(11, 2.2))
for a, img, title in [(ax[0], truth_gev, 'truth'),
                      (ax[1], truth_gev * mask + 1e-3*(1-mask), 'measured (dead box)')]:
    a.imshow(np.clip(img[0].cpu().numpy(), 1e-3, None),
             norm=MplLogNorm(1e-3, 10), aspect='auto', cmap='inferno')
    a.set_title(title)
sl_e, sl_p = slice(ETA0, ETA0+BOX), slice(PHI0, PHI0+BOX)
truth_box = truth_gev[0, sl_e, sl_p].cpu().numpy()
print('truth box [GeV]:'); print(truth_box.round(3))

## 5. Run all five inpainting algorithms

Each draws `N_POST` posterior samples (batched through the reverse process).
Sanity printed per algorithm: the known region must equal `y` **exactly** for
RePaint/DDNM/DDRM/MCG (hard projection); ΠGDM enforces consistency only
through its guidance term.

In [ ]:
from calo_inpaint.inpainting import INPAINTERS

posteriors = {}
for name, cls in INPAINTERS.items():
    inp = cls(net, sched, DEVICE, seed=0, use_bf16=USE_BF16)
    t0 = time.time()
    smp_log = inp.inpaint(y_log, mask, N_POST)          # (N, 1, 24, 64) log
    dt = time.time() - t0
    known_dev = ((smp_log - y_log) * mask).abs().max().item()
    posteriors[name] = lognorm.denormalize(smp_log[:, 0]).cpu().numpy()
    print(f'{name:8s}  {dt:6.1f}s   known-region max|dev| = {known_dev:.2e}')

## 6. Posterior visualization

Per algorithm: posterior mean, posterior std, and one sample — dead box only,
GeV. The event-by-event spread (std) is the scientific target; a smooth mean
with slope<1 vs truth is expected shrinkage, not a defect.

In [ ]:
algs = list(posteriors)
fig, axes = plt.subplots(len(algs), 4, figsize=(10, 2.1*len(algs)))
for r, name in enumerate(algs):
    box = posteriors[name][:, sl_e, sl_p]               # (N, BOX, BOX)
    panels = [truth_box, box.mean(0), box.std(0), box[0]]
    titles = ['truth', f'{name} mean', f'{name} std', f'{name} sample']
    for c, (img, ttl) in enumerate(zip(panels, titles)):
        a = axes[r, c]
        im = a.imshow(img, cmap='inferno'); a.set_title(ttl, fontsize=8)
        a.set_xticks([]); a.set_yticks([])
        fig.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout()

## 7. Mini statistical analysis (SBC / coverage / pulls / TARP)

Small in-notebook version of `scripts/run_statistical_analysis.py`:
`N_IMAGES_MINI` truth images, one algorithm. With few images these are noisy —
the batch script with 1000 images gives the real answer.

In [ ]:
ALG_MINI = os.environ.get('ALG_MINI', 'ddnm')
inp = INPAINTERS[ALG_MINI](net, sched, DEVICE, seed=0, use_bf16=USE_BF16)

all_s, all_t = [], []
t0 = time.time()
for i in range(N_IMAGES_MINI):
    tg = torch.from_numpy(events[i:i+1].copy()).to(DEVICE)
    inp.reseed(1000003 + i)
    s = inp.inpaint(lognorm.normalize(tg), mask, N_POST)
    all_s.append(lognorm.denormalize(s[:, 0, sl_e, sl_p]).cpu().numpy())
    all_t.append(tg[0, sl_e, sl_p].cpu().numpy())
S_ = np.stack(all_s); T_ = np.stack(all_t)     # (N, J, b, b), (N, b, b)
print(f'{N_IMAGES_MINI} images x {N_POST} samples in {time.time()-t0:.1f}s')

rng = np.random.default_rng(0)
logS, logT = np.log(np.clip(S_, 1e-3, None)), np.log(np.clip(T_, 1e-3, None))

# SBC ranks (monotone-invariant)
ranks = (S_ < T_[:, None]).sum(1)
# central coverage
levels = np.arange(0.1, 0.91, 0.1)
cov = [float(((T_ >= np.quantile(S_, (1-g)/2, axis=1)) &
              (T_ <= np.quantile(S_, (1+g)/2, axis=1))).mean()) for g in levels]
# pulls (log space)
z = (logT - logS.mean(1)) / (logS.std(1, ddof=1) + 1e-12)
# TARP (standardized log space, joint over box)
sf = logS.reshape(len(S_), N_POST, -1); tf = logT.reshape(len(T_), -1)
mu, sd = sf.mean((0,1)), sf.std((0,1)) + 1e-12
sf, tf = (sf-mu)/sd, (tf-mu)/sd
lo = np.minimum(sf.min((0,1)), tf.min(0)); hi = np.maximum(sf.max((0,1)), tf.max(0))
ref = lo + (hi-lo) * rng.random((len(S_), sf.shape[-1]))
f = (np.linalg.norm(sf-ref[:,None], axis=2) <
     np.linalg.norm(tf-ref, axis=1)[:,None]).mean(1)
alpha = np.linspace(0, 1, 51); ecp = (f[None] <= alpha[:,None]).mean(1)

fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))
ax[0,0].hist(ranks.ravel(), bins=np.arange(N_POST+2)-0.5, density=True, alpha=0.7)
ax[0,0].axhline(1/(N_POST+1), color='k', ls='--'); ax[0,0].set_title('SBC ranks')
ax[0,1].plot(levels, cov, 'o-'); ax[0,1].plot([0,1],[0,1],'k--')
ax[0,1].set_title('central coverage'); ax[0,1].set_xlabel('nominal')
ax[1,0].hist(np.clip(z.ravel(), -5, 5), bins=40, density=True, alpha=0.7)
g = np.linspace(-5, 5, 200)
ax[1,0].plot(g, np.exp(-g**2/2)/np.sqrt(2*np.pi), 'k--')
ax[1,0].set_title(f'pulls: mean={z.mean():.2f} std={z.std():.2f}')
ax[1,1].plot(alpha, ecp); ax[1,1].plot([0,1],[0,1],'k--')
ax[1,1].set_title(f'TARP (max dev {np.abs(ecp-alpha).max():.3f})')
fig.suptitle(f'{ALG_MINI}, {N_IMAGES_MINI} images — mini calibration check')
plt.tight_layout()

## 8. DDIM η sweep (study extension)

η interpolates deterministic DDIM (η=0) → DDPM-like (η=1). Compare 1×1 tower
spectra across η. (Full sweep: `scripts/generate_events.py --dp ddim --eta ...`)

In [ ]:
etas, spectra = [0.0, 0.5, 1.0], {}
for eta in etas:
    ds = DDIMSampler(net, sched, DEVICE, seed=0, eta=eta, use_bf16=USE_BF16)
    with torch.no_grad():
        g_ = lognorm.denormalize(ds.sample(N_GEN)).squeeze(1).cpu().numpy()
    spectra[eta] = g_

fig, ax = plt.subplots(figsize=(5.5, 3.5))
bins = np.linspace(0, max(s.max() for s in spectra.values()), 50)
for eta, g_ in spectra.items():
    ax.hist(g_.ravel(), bins=bins, histtype='step', density=True,
            label=f'DDIM η={eta}  <ΣE>={g_.sum(axis=(1,2)).mean():.0f} GeV')
ax.set_yscale('log'); ax.set_xlabel(r'$1\times1$ tower $E_T$ [GeV]'); ax.legend()
plt.tight_layout()

## 9. Scaling up

For real numbers, leave the notebook and use the batch pipeline (resumable,
bf16, progress files):

```bash
ROOT=/home/roli/DiffusionProject ./run_all.sh generate verify   # then
ROOT=/home/roli/DiffusionProject ./run_all.sh inpaint stats
```

or individual runs of `scripts/run_inpaint_study.py` in parallel (batch-50
inpainting underutilizes the 5090; 3–5 concurrent runs are fine). Expect
DDRM (η=0.85) to show genuine under-dispersion — intrinsic to its variational
posterior, verified in `tests/test_consistency.py`.